In [0]:
%run "../../utils/00_config"

In [0]:
from pyspark.sql import functions as F

price_quality = spark.table(
    get_gold_table("intermediate", "int_item_price_quality")
)

dim_execution = spark.table(
    get_table("intermediate", "dim_execution")
)

print(f"Price quality records: {price_quality.count():,}")
print(f"Execution records: {dim_execution.count():,}")

In [0]:
# Validate classification values
valid_statuses = [
    "within_iqr",
    "outlier",
    "insufficient_sample"
]

invalid_statuses = (
    price_quality
    .filter(
        F.col("price_quality_status").isNull()
        | ~F.col("price_quality_status").isin(valid_statuses)
    )
    .count()
)

if invalid_statuses > 0:
    raise ValueError(
        f"Found {invalid_statuses} records with invalid quality classifications"
    )

# Aggregate offer quality by execution
quality_summary = (
    price_quality
    .groupBy("execution_id")
    .agg(
        F.count("*").alias("total_eligible_offers"),

        F.sum(
            F.when(F.col("price_quality_status") == "within_iqr", 1).otherwise(0)
        ).alias("within_iqr_offers"),

        F.sum(
            F.when(F.col("price_quality_status") == "outlier", 1).otherwise(0)
        ).alias("outlier_offers"),

        F.sum(
            F.when(F.col("price_quality_status") == "insufficient_sample", 1).otherwise(0)
        ).alias("insufficient_sample_offers")
    )
)

# display(quality_summary)

In [0]:
# Prepare execution metadata
execution_metadata = (
    dim_execution
    .select(
        "execution_id",
        F.col("started_at").alias("execution_started_at"),
        F.col("finished_at").alias("execution_finished_at"),
        "execution_duration_seconds",
        F.col("status").alias("execution_status"),
        F.col("response_count").alias("execution_response_count"),
        F.col("item_count").alias("execution_item_count"),
        F.col("total_orders").alias("execution_total_orders")
    )
)

# Validate execution metadata uniqueness
duplicate_executions = (
    execution_metadata
    .groupBy("execution_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_executions > 0:
    raise ValueError(
        f"Found {duplicate_executions} duplicated execution IDs"
    )

# Validate quality execution coverage
missing_executions = (
    quality_summary
    .select("execution_id")
    .join(
        execution_metadata.select("execution_id"),
        on="execution_id",
        how="left_anti"
    )
    .count()
)

if missing_executions > 0:
    raise ValueError(
        f"Found {missing_executions} executions without metadata"
    )

# Combine quality and execution metrics
rel_market_quality = (
    execution_metadata
    .join(
        quality_summary,
        on="execution_id",
        how="left"
    )
    .fillna(
        0,
        subset=[
            "total_eligible_offers",
            "within_iqr_offers",
            "outlier_offers",
            "insufficient_sample_offers"
        ]
    )
)

print(f"Reporting records: {rel_market_quality.count():,}")

# display(rel_market_quality)

In [0]:
# Calculate collection eligibility metrics
rel_market_quality = (
    rel_market_quality
    .withColumn(
        "eligible_offers_pct",
        F.when(
            F.col("execution_total_orders") > 0,
            F.round(
                F.col("total_eligible_offers")
                / F.col("execution_total_orders") * 100,
                2
            )
        )
    )
    .withColumn(
        "excluded_offers",
        F.col("execution_total_orders") - F.col("total_eligible_offers")
    )
    .withColumn(
        "excluded_offers_pct",
        F.when(
            F.col("execution_total_orders") > 0,
            F.round(
                F.col("excluded_offers")
                / F.col("execution_total_orders") * 100,
                2
            )
        )
    )
    .withColumn(
        "within_iqr_pct",
        F.when(
            F.col("total_eligible_offers") > 0,
            F.round(
                F.col("within_iqr_offers")
                / F.col("total_eligible_offers") * 100,
                2
            )
        )
    )
    .withColumn(
        "outlier_pct",
        F.when(
            F.col("total_eligible_offers") > 0,
            F.round(
                F.col("outlier_offers")
                / F.col("total_eligible_offers") * 100,
                2
            )
        )
    )
    .withColumn(
        "insufficient_sample_pct",
        F.when(
            F.col("total_eligible_offers") > 0,
            F.round(
                F.col("insufficient_sample_offers")
                / F.col("total_eligible_offers") * 100,
                2
            )
        )
    )
)

# display(rel_market_quality)

In [0]:
# Validate market quality metrics
validation = rel_market_quality.agg(
    F.count("*").alias("total_executions"),
    F.sum("total_eligible_offers").alias("eligible_offers"),
    F.sum("within_iqr_offers").alias("within_iqr_offers"),
    F.sum("outlier_offers").alias("outlier_offers"),
    F.sum("insufficient_sample_offers").alias("insufficient_sample_offers"),
    F.sum("excluded_offers").alias("excluded_offers"),

    F.sum(
        F.when(
            F.col("total_eligible_offers") != (
                F.col("within_iqr_offers")
                + F.col("outlier_offers")
                + F.col("insufficient_sample_offers")
            ),
            1
        ).otherwise(0)
    ).alias("invalid_quality_breakdowns"),

    F.sum(
        F.when(
            F.col("execution_total_orders") != (
                F.col("total_eligible_offers") + F.col("excluded_offers")
            ),
            1
        ).otherwise(0)
    ).alias("invalid_eligibility_breakdowns"),

    F.sum(
        F.when(
            (F.col("excluded_offers") < 0)
            | (F.col("total_eligible_offers") < 0),
            1
        ).otherwise(0)
    ).alias("invalid_offer_counts")
)

display(validation)

# Validate required execution metadata
missing_metadata = (
    rel_market_quality
    .filter(
        F.col("execution_id").isNull()
        | F.col("execution_started_at").isNull()
        | F.col("execution_status").isNull()
        | F.col("execution_total_orders").isNull()
    )
    .count()
)

if missing_metadata > 0:
    raise ValueError(
        f"Found {missing_metadata} records with missing execution metadata"
    )

# Validate execution uniqueness
duplicate_reports = (
    rel_market_quality
    .groupBy("execution_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_reports > 0:
    raise ValueError(
        f"Found {duplicate_reports} duplicated execution records"
    )

# Validate metric breakdowns
invalid_breakdowns = (
    validation
    .first()
    .asDict()
)

for column in [
    "invalid_quality_breakdowns",
    "invalid_eligibility_breakdowns",
    "invalid_offer_counts"
]:
    if (invalid_breakdowns[column] or 0) > 0:
        raise ValueError(f"Invalid market quality metric: {column}")

if missing_metadata > 0 or duplicate_reports > 0:
    raise ValueError("Reporting integrity validation failed")

print("Market quality validations passed.")

In [0]:
from delta.tables import DeltaTable

TARGET_TABLE = get_gold_table(
    "reporting",
    "rel_market_quality"
)

merge_condition = """
    target.execution_id = source.execution_id
"""

if spark.catalog.tableExists(TARGET_TABLE):
    (
        DeltaTable.forName(spark, TARGET_TABLE)
        .alias("target")
        .merge(
            rel_market_quality.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        rel_market_quality
        .write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(TARGET_TABLE)
    )

print(f"Table successfully materialized: {TARGET_TABLE}")

In [0]:
# Validate materialized reporting table
saved_quality = spark.table(TARGET_TABLE)

expected_count = rel_market_quality.count()
actual_count = saved_quality.count()

duplicate_count = (
    saved_quality
    .groupBy("execution_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if expected_count != actual_count:
    raise ValueError(
        f"Record count mismatch: expected {expected_count}, found {actual_count}"
    )

if duplicate_count > 0:
    raise ValueError(
        f"Found {duplicate_count} duplicated execution records"
    )

print(f"Saved records: {actual_count:,}")
print(f"Duplicated execution records: {duplicate_count:,}")

# display(saved_quality)